# Converting structures: CIF, ASE and DiffPy

This notebook demonstrates the interop adapters.  It requires the extras
(`pip install 'discus-python[interop]'`); without them the conversion cells
print a skip message and the notebook still runs.

See [docs/INTEROP.md](../docs/INTEROP.md) for the full documentation.

In [1]:
from pathlib import Path
import os
import sys
sys.path.insert(0, str(Path.cwd().parent))
os.chdir(Path.cwd().parent)

import numpy as np
from discuspdf import Lattice, Structure
from discuspdf.cif_io import read_cif, write_cif

FIXTURES = Path('tests/fixtures/interop')

## Read a real CIF (COD GaAs)

In [2]:
# read_cif — read a CIF with gemmi and expand the asymmetric unit by the space group (needs the cif extra)
#   path            CIF file
#   block           data block name; default None = the only block (several blocks raise InteropError)
#   model           "average" (default) or "configuration" (all occupancies must be 1.0);
#                   a CIF written by write_cif restores its own stored model
#   allow_loss      False (default): anisotropic ADPs, disorder groups, magnetic moments raise InteropError;
#                   True warns (InteropWarning) and lists them in .loss
#   min_separation  Å, overlap check (default 0.1; currently honoured only for CIFs written by write_cif)
#   returns         CifImport: .structure (unit cell, supercell (1, 1, 1)), .block, .spacegroup,
#                   .n_asymmetric, .symmetry_source, .loss, .defaults_applied; B from B_iso or 8π²·U_iso
imp = read_cif(FIXTURES / 'gaas_cod_9008845.cif')
s = imp.structure
print(f'CIF: {imp.block}, space group {imp.spacegroup}, '
      f'{imp.n_asymmetric} asymmetric sites -> {s.n_atoms} atoms')
assert s.n_atoms == 8
a = s.lattice.parameters[0]
print(f'a = {a:.4f} A')

CIF: 9008845, space group F -4 3 m, 2 asymmetric sites -> 8 atoms
a = 5.6537 A


## Build a 4x4x4 configuration supercell

`Structure.lattice` is the whole supercell frame (unit cell x 4).

In [3]:
n = (4, 4, 4)
m = s.n_atoms
# Lattice.from_parameters — lattice from the six cell parameters (canonical: a ∥ x, b in the xy plane)
#   a, b, c              lengths in Å (finite, > 0)
#   alpha, beta, gamma   angles in degrees, strictly between 0 and 180
#                        (alpha = angle b–c, beta = angle a–c, gamma = angle a–b)
lat = Lattice.from_parameters(a * n[0], a * n[1], a * n[2], *s.lattice.parameters[3:])
# Supercell atoms in DISCUS order (cell-major: all m atoms of cell 0, then cell 1, …)
#   cell_num   which unit cell atom i belongs to, 0 … n1·n2·n3 − 1 (x fastest, then y, then z)
#   slot       which site of the unit cell atom i is, 0 … m − 1 (the row in the unit-cell file);
#              it picks that site's species and fractional position from s
N = m * n[0] * n[1] * n[2]
cell_num = np.arange(N) // m
slot = np.arange(N) % m
cell_index = np.stack(
    [cell_num % n[0], (cell_num // n[0]) % n[1], cell_num // (n[0] * n[1])],
    axis=1,
).astype(np.int64)
# Structure parameters (lengths in Å, B in Å²)
#   lattice          Lattice of the WHOLE periodic frame (for a supercell, the supercell lattice)
#   species, frac    (N,) labels (non-empty, ≤ 8 characters, no whitespace) and (N, 3) fractional coordinates
#   model            required: "configuration" (realised, every occupancy 1.0) or "average" (site occupancies)
#   occupancy        (N,) values in (0, 1]; default all 1.0
#   b_iso            (N,) isotropic B ≥ 0, or None (default)
#   cell_index       (N, 3) unit-cell index of each atom, 0 ≤ index < supercell; default all 0
#   supercell        (n1, n2, n3) ints ≥ 1; default (1, 1, 1)
#   pbc              three booleans; default (True, True, True)
#   site_label       default species + 1-based count per species (e.g. GA1); atom_id default 0..N-1
#   min_separation   Å; atoms in different sites closer than this raise ValueError (default 0.1; None skips)
supercell = Structure(
    lat,
    s.species[slot],
    (s.frac[slot] + cell_index) / np.array(n, dtype=float),
    model='configuration',
    cell_index=cell_index,
    supercell=n,
    pbc=(True, True, True),
)
print(f'4x4x4 configuration supercell: {supercell.n_atoms} atoms, '
      f'supercell {tuple(int(x) for x in supercell.supercell)}')
assert supercell.n_atoms == 512

4x4x4 configuration supercell: 512 atoms, supercell (4, 4, 4)


## Nearest-neighbour check

In zinc blende every atom has 4 neighbours of the other species at
a*sqrt(3)/4.  The notebook fails loudly if that is wrong.

In [4]:
# Structure.distance_matrix — (M, M) interatomic distances in Å (minimum image when periodic)
#   ids          atom_id values to include (default None = all atoms); M > 5000 is refused
D = supercell.distance_matrix()
np.fill_diagonal(D, np.inf)
d_min = D.min()
expected = a * np.sqrt(3) / 4
assert abs(d_min - expected) < 1e-6, f'min distance {d_min:.6f} != {expected:.6f}'
nn = (D < 2.6).sum(axis=1)
assert set(nn.tolist()) == {4}, sorted(set(nn.tolist()))
print(f'nearest-neighbour check: min distance {d_min:.6f} A '
      f'(expected {expected:.6f}), 4 neighbours per atom')

nearest-neighbour check: min distance 2.448124 A (expected 2.448124), 4 neighbours per atom


## Convert to ASE and DiffPy

These cells run only if the extras are installed; otherwise they print a
skip message.

In [5]:
try:
    from discuspdf.ase_io import to_ase, from_ase
    # to_ase — Structure → ase.Atoms (needs the ase extra); the full state is kept in dp_* arrays/info keys
    #   structure    any Structure; an average model with partial or mixed sites gives an InteropWarning
    atoms = to_ase(s)
    # from_ase — ase.Atoms → Structure; Atoms made by to_ase are restored exactly from the dp_* keys
    #   model           None (default) or "configuration"/"average"; required for a foreign Atoms object
    #   allow_loss      False (default): unsupported features of a foreign Atoms (constraints, calculator,
    #                   tags, charges, magnetic moments, ...) raise InteropError; True warns and drops them
    #   min_separation  Å, overlap check (default 0.1)
    s2 = from_ase(atoms)
    # Structure.isclose — True if two Structures agree
    #   rtol, atol   tolerances on lattice, frac, occupancy and b_iso (defaults 1e-12, 1e-15); species,
    #                site_label, atom_id, cell_index, supercell, pbc and model must match exactly
    assert s2.isclose(s, rtol=1e-8, atol=1e-12)
    print(f'ASE round trip: OK ({len(atoms)} atoms, dp_* arrays attached)')
except ImportError as e:
    print(f'ASE skipped: {e}')

try:
    from discuspdf.diffpy_io import to_diffpy, from_diffpy
    # to_diffpy — Structure → diffpy.structure.Structure (needs the diffpy extra); orientation is not kept
    #   structure    B is stored as Uisoequiv = B / 8π²; the model is not stored (pass it to from_diffpy)
    #   allow_loss   False (default): atom_id not 0..N-1, non-trivial cell_index/supercell, pbc not all True
    #                or charged species raise InteropError; True warns and drops them
    ds = to_diffpy(s)
    # from_diffpy — diffpy.structure.Structure → Structure (lattice from a, b, c, alpha, beta, gamma)
    #   model        required: "configuration" or "average"
    #   allow_loss   False (default): anisotropic atoms raise InteropError; True keeps the isotropic equivalent
    #   B = 8π²·Uisoequiv (None if all zero); atom_id 0..N-1, supercell (1, 1, 1), pbc all True
    s3 = from_diffpy(ds, model=s.model)
    assert s3.isclose(s, rtol=1e-8, atol=1e-12)
    print(f'DiffPy round trip: OK ({len(ds)} atoms)')
except ImportError as e:
    print(f'DiffPy skipped: {e}')

ASE round trip: OK (8 atoms, dp_* arrays attached)
DiffPy round trip: OK (8 atoms)


## Write a DISCUS .cll and a CIF

In [6]:
import tempfile
with tempfile.TemporaryDirectory() as tmpdir:
    cll_path = Path(tmpdir) / 'gaas.cll'
    cif_path = Path(tmpdir) / 'gaas.cif'
    from discuspdf.discus_io import write_cll, read_cll
    # write_cll — write a unit cell as a DISCUS .cll file (atomic write, then parsed back and checked)
    #   path         target file; an existing file is refused unless overwrite=True (default False)
    #   obj          Structure (supercell (1, 1, 1), all pbc True) or DiscusCell
    #   title        `title` line (default "structure", or the DiscusCell's own title)
    #   spacegroup   default "P1" for a Structure (atoms written explicitly), or the DiscusCell's own
    #   B is always written (1.0 Å² when b_iso is None; B < 0.001 Å² is refused); 6 decimals
    #   returns      the target Path
    write_cll(cll_path, s)
    # write_cif — write a Structure as a P1 CIF with _discuspdf_* tags for exact round trip (atomic, checked)
    #   path         target file; an existing file is refused unless overwrite=True (default False)
    #   structure    any Structure (model, supercell, site labels, atom ids, cell_index are stored)
    #   block_name   data block name (default "discuspdf")
    #   allow_loss   False (default): mixed pbc raises InteropError; True warns (reader assumes all periodic)
    #   returns      the target Path
    write_cif(cif_path, s)
    # read_cll — read a DISCUS .cll unit-cell source (DISCUS `read cell`); a .stru file is rejected
    #   path         .cll file
    #   returns      DiscusCell: .title, .spacegroup, .structure, .defaults_applied
    #                .structure holds the atoms AS LISTED (not expanded by the space group); a missing or
    #                zero B becomes 1.0 Å² and "b_iso" is recorded in .defaults_applied
    s_cll = read_cll(cll_path).structure
    # .cll has 6-decimal precision and regenerates site labels; compare
    # lattice, coordinates, occupancy and B factor (not the exact labels).
    assert np.allclose(s_cll.lattice.parameters, s.lattice.parameters, atol=1e-6)
    assert np.allclose(s_cll.frac, s.frac, atol=1e-6)
    assert np.allclose(s_cll.occupancy, s.occupancy, atol=1e-6)
    # .cll has no B column here, so read_cll defaults B to 1.0; skip the B
    # comparison for this B-free fixture.
    s_cif = read_cif(cif_path).structure
    assert s_cif.isclose(s, rtol=1e-8, atol=1e-9)
    print(f'.cll and CIF written and read back: {s_cll.n_atoms} atoms each')
    print('--- gaas.cll (first 6 lines) ---')
    for line in cll_path.read_text().splitlines()[:6]:
        print(line)

.cll and CIF written and read back: 8 atoms each
--- gaas.cll (first 6 lines) ---
title structure
spcgr P1
cell 5.653700 5.653700 5.653700 90.000000 90.000000 90.000000
atoms
Ga   0.000000 0.000000 0.000000 1.000000
Ga   0.000000 0.500000 0.500000 1.000000


## Summary

* `read_cif` (gemmi) expands the asymmetric unit by the space group.
* `to_ase` / `from_ase` round-trip exactly via `dp_*` arrays.
* `to_diffpy` / `from_diffpy` preserve everything except orientation, site
  labels, atom ids, cell indices, supercell, pbc and the model.
* `write_cif` emits `_discuspdf_*` tags for exact round-trip.